# Main Colab — Phase A starter, chưa là notebook inference hoàn chỉnh
Chọn ROLE đúng. TV2/TV3 phải hoàn thiện phần platform được giao, tự tests/commit/PR.
TV1 sở hữu sharding; TV4 sở hữu export/recovery/collection. Mọi người dùng một core runner.
Notebook không chứa outputs/tokens. Chưa có cloud generation đã được xác minh.
Chỉ Phase B sau khi bốnPR tích hợp, pinned commit/release và freeze v2 được duyệt mới chạy main.


In [ ]:
from pathlib import Path
import os, sys, subprocess
ROLE = "TV1"  # Colab: TV1/TV3; Kaggle: TV2/TV4.
BUNDLE_ROOT = Path("/content/VLM_ENVI_Main_Inference_Team_v1")
PROJECT = BUNDLE_ROOT / "project"  # Hoặc clone chung đã checkout đúng base/commit.
assert PROJECT.is_dir(), "Giải nén master đã kiểmSHA hoặc chỉ đến Git clone của bạn."
assert ROLE in ['TV1', 'TV3'], "Sai role/platform"
ASSIGNMENT = PROJECT / "configs/team_assignments_v1" / (ROLE + ".yaml")
assert ASSIGNMENT.is_file()


## Setup (explicit)
Bật GPU trong UI, kiểm Internet. Chuẩn bị assets ZIP riêng tư. Không đổi requirements để ép chạy.
Đặt INSTALL=True khi muốn cài bằng setup hiện có; dùng venv riêng để không phá packages của notebook.
TV2/TV3 bổ sung platform checks và bằng chứng ở PR của mình.


In [ ]:
INSTALL = False
ENV = PROJECT / ".venv-cloud"
if INSTALL:
    subprocess.run([sys.executable, "scripts/setup_inference.py", "--env", str(ENV)], cwd=PROJECT, check=True)
PYTHON = ENV / "bin/python"
assert PYTHON.is_file(), "Chạy setup với INSTALL=True trước; trên Windows dùng Scripts/python.exe."


In [ ]:
ASSETS = BUNDLE_ROOT / "main_assets/main_assets_v1.zip"
assert ASSETS.is_file(), "Attach/private assets ZIP hoặc đặt đúng path; không dùng weights ZIP."
subprocess.run([str(PYTHON), "scripts/main_team_bundle.py", "restore-assets", "--archive", str(ASSETS), "--root", str(PROJECT)], cwd=PROJECT, check=True)
subprocess.run([str(PYTHON), "scripts/team_assignment.py", "plan", "--assignment", str(ASSIGNMENT)], cwd=PROJECT, check=True)


In [ ]:
# Gemma: tài khoản của chính bạn đã accept license, secret HF_TOKEN được cấp cho notebook.
MODEL = "gemma" if ROLE in ("TV3", "TV4") else "qwen"
if MODEL == "gemma":
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# Không print token/env hoặc lưu notebook outputs chứa secrets.
probe = "import json,torch; from vlmbench.pilot_launcher import gpu_preflight; print(json.dumps(gpu_preflight('" + MODEL + "',torch)))"
subprocess.run([str(PYTHON), "-c", probe], cwd=PROJECT, check=True)


## Phần TV2/TV3 phải hoàn thiện và kiểm thật
- Dataset mount/upload/version/path validation theo platform; setup/Internet/Secrets error handling.
- Diagnostics package fingerprint, CUDA/VRAM, native BF16 nếu Gemma. Probe trên không load model.
- Restore checkpoint và export backup tích hợp với TV4; không tự xóa checkpoint khi lỗi.
- Ghi evidence thật, clear outputs trước commit notebook.

## Phase B — chưa được bật
Sau khi TV1 merge --language và TV4 merge export/collection, TV5 chốt cùng40hexcommit,
implementation/protocol v2 và execution approval cho bốnshard. Chạy smoke rồi mới1.000cells/shard.
Các lệnh cụ thể trong members/<role>/RUN_GUIDE.md là lệnh mục tiêu sauintegration, không phải tính năng đã có.


In [ ]:
PHASE_B_APPROVED = False
if PHASE_B_APPROVED:
    raise RuntimeError("TV5 phải thay cell này bằng verified shared release workflow sau PR integration; không bật bằng cách đổi boolean.")
print("Phase A only: no inference calls. Development/real cloud evidence still belongs to the assigned member.")
